# audstim recipes

Install once from the repo root: `pip install -e ".[notebook,sofa]"`

Any `Sound` left as the last expression in a cell shows up as an audio player.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from tqdm.auto import tqdm

import audstim as au
from audstim import dB
from audstim.utils import time_axis

FS = 44100
rng = np.random.default_rng(2026)  # one seeded generator -> reproducible stimuli

SOUNDS = Path("~/Sync/Sounds").expanduser()
QA_DIR = SOUNDS / "QACorpus"
HRIR_DIR = SOUNDS / "HRTF-PKU-IOA" / "all_HRIRs"


def qa(talker: str, sentence: str = "Which is bigger, a moose or a bee ") -> au.Sound:
    """Load a QA-corpus sentence by talker ID, e.g. qa("F02")."""
    return au.load(QA_DIR / f"{talker} {sentence}.wav").mono()

# Levels in dB
`6*dB` is a unit value: adding it to a Sound changes the level, adding a Sound mixes,
and adding a bare number is an error (it would be ambiguous with a DC offset).

In [ ]:
tone = au.pure_tone(1.0, FS, 440).ramp(10e-3)
noise = au.gaussian_noise(1.0, FS, rng=rng)

louder = tone + 6 * dB
softer = tone - 10 * dB
snr = -5
mixture = tone + (noise - snr * dB)  # tone at -5 dB SNR re the noise

au.relative_db([tone, louder, softer, noise - snr * dB])

# Generate impulse response
Independent tails in each ear give a decorrelated (diffuse) binaural reverb.
`decay_db=20` matches the old `dB_thresh=-20`.

In [ ]:
ir = au.synth_ir(rt60=0.5, fs=FS, decay_db=20, n_channels=2, rng=rng).normalize(peak=1)
ir.plot()
ir

In [ ]:
# With a direct path at a given direct-to-reverberant ratio
ir_drr = au.synth_ir(rt60=0.5, fs=FS, drr_db=5, rng=rng)
print(f"measured DRR: {10 * np.log10(ir_drr.data[0, 0] ** 2 / np.sum(ir_drr.data[1:, 0] ** 2)):.1f} dB")

To do:
* Incorporate distance cues

For a point source, intensity falls as $\frac{1}{4\pi r^2}$, so every doubling of distance is
**−6 dB** (−3 dB per doubling is the line-source / cylindrical-spreading case).
`au.distance_gain_db(r, ref=1.0)` gives $-20\log_{10}(r/r_\text{ref})$.
Check whether your HRIR set already includes distance attenuation before applying it.

In [ ]:
print(au.distance_gain_db([0.5, 1, 2, 4]))
far = tone + au.distance_gain_db(4.0) * dB  # the tone as if moved from 1 m to 4 m
far

# Basic stimuli

In [ ]:
dur = 1.0

# x = au.gaussian_noise(dur, FS, band=(200, 2000), tilt=-3, rng=rng)  # tilt is dB/octave now
# x = au.pure_tone(dur, FS, 1000)
# x = au.pulse_train(dur, FS, 1000, phase=np.pi / 2)
# x = au.square_wave(dur, FS, 1000)          # band-limited; bandlimited=False for the naive one
# x = au.sawtooth_wave(dur, FS, 1000)
# x = au.linear_chirp(dur, FS, 16000, 1000)
# x = au.exponential_chirp(dur, FS, 1000, 16000)
# x = au.schroeder_complex(dur, FS, 100)     # all harmonics below Nyquist

x = au.harmonic_complex(
    dur,
    FS,
    f0=100,
    harmonics=[1, 2, 3],
    amplitudes=[1, 0.6, 0.3],
    phases=[0, np.pi / 4, -np.pi / 3],
)
au.overview(x, win_dur=20e-3)
x

# Make speech-shaped noise
Averages Welch spectra across talkers (a smooth long-term average spectrum) instead of
summing waveforms and using one raw FFT.

In [ ]:
talkers = sorted(QA_DIR.glob("M*.wav"))
ltass = au.long_term_spectrum([au.load(p) for p in tqdm(talkers)])
ssn = ltass.to_noise(3, FS, rng=rng).ramp(20e-3)

fig, ax = plt.subplots()
ltass.smooth(1 / 3).plot(ax, label="LTASS (1/3-oct smoothed)")
au.long_term_spectrum(ssn).plot(ax, label="speech-shaped noise", alpha=0.7)
ax.legend()
ssn

# Generate the sum of randomly generated sounds

In [ ]:
n_sounds, dur = 100, 2.0
f_nyq = FS / 2

sounds = [
    # au.pure_tone(dur, FS, rng.uniform(20, f_nyq), rng.uniform(-np.pi, np.pi))
    # au.sawtooth_wave(dur, FS, rng.uniform(20, f_nyq), rng.uniform(-np.pi, np.pi))
    au.exponential_chirp(dur, FS, *rng.uniform(20, f_nyq, 2), phase=rng.uniform(-np.pi, np.pi))
    for _ in tqdm(range(n_sounds))
]
chirp_sum = sum(sounds).zero_mean().normalize().ramp(10e-3)
au.overview(chirp_sum)
chirp_sum

# STFT of some sentences

In [ ]:
target, masker = au.normalize(au.pad(au.match_fs([qa("F02"), qa("M08")]), align="center"))
mixture = target + (masker + 6 * dB)  # masker 6 dB above target

S_target, S_masker, S_mixture = (au.STFT(s, win_dur=20e-3) for s in (target, masker, mixture))

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, layout="constrained")
titles = ("Target", "Masker (+6 dB)", "Mixture")
for S, ax, title in zip((S_target, S_masker, S_mixture), axes, titles, strict=True):
    S.plot(ax, fmax=8000, colorbar=False)
    ax.set_title(title)
mixture

# Noise vocoding example
One-liner, then the same thing spelled out. Filters are power-complementary, so
analysis → synthesis is exact.

In [ ]:
target = qa("F02")
vocoded = au.noise_vocode(target, n_bands=16, carrier="noise", env_lowpass=50, rng=rng)
au.overview(vocoded, fmax=8000)
vocoded

In [ ]:
sb = au.subbands(target, n_bands=16, f_lo=80, f_hi=8000)

# Choose among carriers (must be at least as long as the target)
carrier = au.gaussian_noise(target.duration, target.fs, rng=rng)
# carrier = au.pulse_train(target.duration, target.fs, 100)
# carrier = sum(au.exponential_chirp(target.duration, target.fs, *rng.uniform(20, 8000, 2))
#               for _ in range(100))

carrier_tfs = sb.filterbank.analyze(carrier).tfs()
vocoded = (sb.envelopes(lowpass=50) * carrier_tfs).synthesize().normalize(target.rms)

sb.envelopes().plot()
vocoded

# Ideal binary mask reconstruction
Fixed: the STFT now has an exact inverse, so masking and resynthesis work.

In [ ]:
ibm = au.ideal_binary_mask(S_target, S_masker, lc_db=0)  # lc_db replaces the old "masker_STFT + 0"
separated = (S_mixture * ibm).to_sound()

# soft alternative: au.ideal_ratio_mask(S_target, S_masker)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True, layout="constrained")
ibm.plot(axes[0])
au.STFT(separated).plot(axes[1], fmax=8000, colorbar=False)
axes[0].set_ylim(0, 8)
axes[1].set_title("IBM-processed mixture")
separated

# Generate ITD/ILD and inspect subband representation
Sign convention: positive ITD = right ear leads, positive ILD = right ear louder.

In [ ]:
int_dur, sil_dur, ramp_dur = 0.4, 0.1, 25e-3

noise = au.correlated_noise(int_dur, FS, corr=1, rng=rng).ramp(ramp_dur)
# noise = au.schroeder_complex(int_dur, FS, 200, n_harmonics=1000).ramp(ramp_dur).to_stereo()

interval1 = noise.convolve(au.simple_bir(FS, itd=-500e-6, ild=3))
interval2 = noise.convolve(au.simple_bir(FS, itd=500e-6, ild=-3))
stim = au.concat([interval1, au.silence(sil_dur, FS), interval2]).normalize()

stim.plot()
au.interaural_cues(stim, win_dur=50e-3).plot()
au.interaural_cues(stim, win_dur=20e-3, filterbank=au.ERBFilterbank(16, 100, 8000)).plot()
au.subbands(stim, n_bands=12, f_lo=100, f_hi=8000).envelopes().plot()
stim

# Make a moving sound
Each trajectory point gets a raised-cosine cross-fade window, and HRIRs are interpolated
with onsets aligned (delays interpolated separately), which avoids comb filtering.

In [ ]:
hrirs = au.HRIRSet.from_pku_ioa(HRIR_DIR)
# hrirs = au.HRIRSet.from_sofa(SOUNDS / "some_database.sofa")

speech = qa("F02")
# speech = au.gaussian_noise(2, FS, rng=rng)

traj = au.circular_trajectory((100, 0, 90), (100, 0, -90), 500)  # right -> front -> left at 1 m
# traj = au.linear_trajectory((0.5, 0.5, 0), (-0.5, 0.5, 0), 500)  # x = right, y = front, z = up

moving = au.move_sound(speech, traj, hrirs)
au.interaural_cues(moving, win_dur=20e-3).plot()
moving

# Moving sound with synthetic IR
The old version reverberated the dry sound and *then* moved it, which also moves the
reverb. Here the direct path follows the trajectory and a decorrelated late tail is
added at a chosen DRR. (A position-dependent BRIR would also move the early reflections.)

In [ ]:
traj = au.linear_trajectory((0.5, 0.5, 0), (-0.5, 0.5, 0), 500)
direct = au.move_sound(speech, traj, hrirs)

tail = au.synth_ir(rt60=1.0, fs=speech.fs, decay_db=30, n_channels=2, rng=rng)
late = speech.convolve(tail)

drr_db = 6.0
late = late * np.sqrt(np.sum(direct.data**2) / np.sum(late.data**2)) - drr_db * dB
reverberant = au.mix([direct, late])

fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
tail.plot(axes[0])
reverberant.plot(axes[1])
au.interaural_cues(reverberant, win_dur=20e-3).plot()
reverberant

# Generating amplitude modulated tone and binaural beat stimulus
See Rutschmann and Rubinstein (1965), "Binaural Beats and Binaural Amplitude‐Modulated
Tones: Successive Comparison of Loudness Fluctuations."

In [ ]:
dur, f0, f1, m = 5, 440, 2, 2
t = time_axis(dur * FS, FS)

y = np.sin(2 * np.pi * f0 * t)
y_AM = (1 + m * np.sin(2 * np.pi * f1 * t)) * np.sin(2 * np.pi * f0 * t)
# NB: as written, y_o is algebraically identical to y_AM (carrier + both sidebands)
y_o = y + (m / 2) * np.cos(2 * np.pi * (f0 - f1) * t) - (m / 2) * np.cos(2 * np.pi * (f0 + f1) * t)

am_tone = au.Sound(y_AM, FS)
au.overview(am_tone)
display(am_tone)

binaural_beats = au.Sound.from_channels(y, y_o, fs=FS)
binaural_beats

# Heller stimulus (AM in late ear)

In [ ]:
dur, f0, itd, ramp_dur = 3, 200, 1e-3, 50e-3

right = au.harmonic_complex(dur, FS, f0, harmonics=np.arange(1, 100)).ramp(ramp_dur)
# right = au.gaussian_noise(dur, FS, rng=rng).ramp(ramp_dur)
left = au.amplitude_modulate(right, f_mod=4, depth=0.5).ramp(ramp_dur)

tremolo_transfer = au.Sound.from_channels(left.delay(itd), right, fs=FS).ramp(ramp_dur)
tremolo_transfer

# Generate iterated ripple noise
See Yost (1996) "Pitch of iterated ripple noise."

In [ ]:
irn = au.iterated_ripple_noise(1, FS, delay=8e-3, gain=1, iterations=16, rng=rng)
au.overview(irn, win_dur=50e-3)
irn

# Generate Oscor (oscillating correlation)
See Siveke et al. (2008) "Psychophysical and Physiological Evidence for Fast Binaural Processing."

The bottom panel shows the zero-lag correlation following sin(2π·f_mod·t).

In [ ]:
oscor = au.oscor(5, FS, f_mod=3, rng=rng)
au.interaural_cues(oscor, win_dur=10e-3).plot()
oscor

# Generate Phasewarp
See Siveke et al. (2008) "Psychophysical and Physiological Evidence for Fast Binaural Processing."

Why the old version "didn't seem to work": the phases were drawn with
`np.random.normal` (±1 rad around 0) rather than uniformly on the circle, so the "noise"
was mostly a click, and the IAC was never computed so you couldn't see the effect.
The rolling idea itself was right. `au.phasewarp` does the equivalent single-sideband
frequency shift; the zero-lag correlation now follows cos(2π·f_mod·t).

In [ ]:
phasewarp = au.phasewarp(5, FS, f_mod=5, rng=rng)
au.interaural_cues(phasewarp, win_dur=10e-3).plot()
phasewarp

# Generate rough, beating, and non-beating tones

In [ ]:
def beating_pair(f, df, dur=4.0):
    return (au.pure_tone(dur, FS, f) + au.pure_tone(dur, FS, f + df, phase=np.pi / 2)).ramp(50e-3)


for f in (440, 440 * 4):
    for df in (6, 20, 40, 110):
        pair = beating_pair(f, df)
        print(f"{f} + {f + df} Hz")
        display(pair)
        # pair.normalize(peak=0.9).save(Path(f"~/Downloads/{f}_{f + df}.mp3").expanduser())

# Generate random tone sequences

In [ ]:
def tone_sequence(freqs, tone_dur, gap_dur, ramp_dur=10e-3):
    """Tones at `freqs` separated by gaps, with a leading gap."""
    parts = [au.silence(gap_dur, FS)]
    for f in freqs:
        parts += [
            au.pure_tone(tone_dur, FS, f, rng.uniform(-np.pi, np.pi)).ramp(ramp_dur),
            au.silence(gap_dur, FS),
        ]
    return au.concat(parts)


erb_lo, erb_hi = au.freq_to_erb([80, 4000])
n_streams, n_tones = 3, 4
streams = [
    tone_sequence(au.erb_to_freq(rng.uniform(erb_lo, erb_hi, n_tones)), tone_dur=0.8, gap_dur=0.2)
    for _ in range(n_streams)
]
summed = sum(streams)
au.overview(summed, win_dur=25e-3)
summed

In [ ]:
# Streams with frequencies clustered (Gaussian in ERB) around a random center each
erb_lo, erb_hi = au.freq_to_erb([100, 8000])
n_streams, n_tones, erb_sd = 3, 50, 0.1
centers = rng.uniform(erb_lo, erb_hi, n_streams)
streams = [
    tone_sequence(au.erb_to_freq(rng.normal(c, erb_sd, n_tones)), tone_dur=0.08, gap_dur=0.02)
    for c in centers
]
summed = sum(streams)
au.overview(summed, win_dur=25e-3)
summed